<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 6 · Solutions</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">How coding assistants work · Solutions</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">Worked answers to the chapter's three practice challenges.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

Worked answers to Chapter 6's practice challenges. The first runs the hand-built loop, and the other two read the build transcripts and the package with no API call.

## Setup

The same Setup as the chapter's notebook, with this notebook's own recording, then the package, the tools and the build transcripts.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/how-coding-assistants-work--solutions.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("how-coding-assistants-work--solutions", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 2 recorded responses from claude-sonnet-5-5, recorded 2026-11-12. Set LIVE = True in Setup to call the API yourself.


In [2]:
# The supportdesk package from this chapter's build: the modules Claude Code generated in Chapters 1, 3, 4
# and 5, and desk.py, which the book wrote to join them. Colab fetches the files from the book's repository.
BUILD = "how-coding-assistants-work"
FILES = ["__init__.py", "ask.py", "desk.py", "retry.py", "tokencost.py", "triage.py"]
if "google.colab" in sys.modules:
    PACKAGE = pathlib.Path("supportdesk")
    PACKAGE.mkdir(exist_ok=True)
    for name in FILES:
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/{BUILD}/given/supportdesk/{name}", PACKAGE / name)
else:
    PACKAGE = pathlib.Path("../../builds", BUILD, "given", "supportdesk")
for name in FILES:
    print(f"{name:13} {len((PACKAGE / name).read_text().splitlines()):>3} lines")

__init__.py     1 lines
ask.py         99 lines
desk.py        32 lines
retry.py       72 lines
tokencost.py   72 lines
triage.py      36 lines


In [3]:
import json

MODEL = "claude-sonnet-5-5"

# Two tools, described to the model as JSON schemas. The model asks for them, and this code runs them.
TOOLS = [
    {"name": "list_files", "description": "List the Python files in the supportdesk package.",
     "input_schema": {"type": "object", "properties": {}}},
    {"name": "read_file", "description": "Return the text of one file in the supportdesk package, such as desk.py.",
     "input_schema": {"type": "object", "properties": {"name": {"type": "string", "description": "The file name"}},
                      "required": ["name"]}},
]


def run_tool(name, args):
    # The tools can reach the package's own files and nothing else
    files = sorted(p.name for p in PACKAGE.glob("*.py"))
    if name == "list_files":
        return "\n".join(files)
    if name == "read_file" and args.get("name") in files:
        return (PACKAGE / args["name"]).read_text()
    return f"error: no file named {args.get('name')!r} in the package"


SYSTEM = ("You answer questions about the Python package supportdesk. Use the tools to read the code before you "
          "answer, and name the functions your answer relies on.")

In [4]:
# The kept transcripts of the book's six Claude Code builds so far
SLUGS = ["what-building-ai-with-ai-means", "how-a-language-model-produces-text", "tokens-context-and-cost",
         "your-first-model-call", "reading-a-response", "how-coding-assistants-work"]
transcripts = {}
for slug in SLUGS:
    if "google.colab" in sys.modules:
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/{slug}/transcript.json", f"{slug}.json")
        path = pathlib.Path(f"{slug}.json")
    else:
        path = pathlib.Path("../../builds", slug, "transcript.json")
    transcripts[slug] = json.loads(path.read_text())

print(f"{'chapter':>7} {'tool calls':>10}  {'refused':>7} {'turns':>5} {'out':>5} {'dollars':>7}   tools in order")
for n, (slug, t) in enumerate(transcripts.items(), 1):
    calls = [s["tool"] for s in t["steps"] if s.get("tool")]
    refused = sum(1 for s in t["steps"] if s.get("denied"))
    r = t["result"]
    print(f"{n:>7} {len(calls):>10}  {refused:>7} {r['turns']:>5} {r['output_tokens']:>5} {r['cost_usd']:>7.4f}   "
          + " ".join(calls))

chapter tool calls  refused turns   out dollars   tools in order
      1          4        0     5   777  0.0140   Read Bash Edit Bash
      2          4        0     5  1049  0.0205   Read Bash Edit Bash
      3          4        0     5  1501  0.0285   Read Bash Edit Bash
      4          5        0     6  1843  0.0357   Read Bash Bash Edit Bash
      5          5        0     6  1337  0.0323   Read Bash Read Edit Bash
      6         11        1    12  3176  0.0648   Read Bash Bash Read Read Read Read Read Read Edit Bash


In [5]:
import sys as _sys
_sys.path.insert(0, str(PACKAGE.parent))
from supportdesk import triage

## Challenge 1. A loop on a budget

Run the same loop with a limit of two requests and a question that needs most of the package read, and see what comes back.

In [6]:
# Challenge 1. The same loop with a budget of two requests and a question that needs more reading.
QUESTION = "Which modules in supportdesk call the Messages API directly, and which only through another module?"
messages = [{"role": "user", "content": QUESTION}]
for turn in range(1, 3):
    reply = client.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM, tools=TOOLS, messages=messages)
    messages.append({"role": "assistant", "content": reply.content})
    calls = [block for block in reply.content if block.type == "tool_use"]
    print(f"turn {turn}: stop_reason {reply.stop_reason!r}, tools {[(c.name, c.input) for c in calls]}")
    if reply.stop_reason != "tool_use":
        break
    messages.append({"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": c.id, "content": run_tool(c.name, c.input)} for c in calls]})
text = "".join(block.text for block in reply.content if block.type == "text")
print("\nanswer:", repr(text) if text else "none, the budget ran out while the model was still reading")

turn 1: stop_reason 'tool_use', tools [('list_files', {})]
turn 2: stop_reason 'tool_use', tools [('read_file', {'name': '__init__.py'}), ('read_file', {'name': 'ask.py'}), ('read_file', {'name': 'desk.py'}), ('read_file', {'name': 'retry.py'}), ('read_file', {'name': 'tokencost.py'}), ('read_file', {'name': 'triage.py'})]

answer: none, the budget ran out while the model was still reading


The first request listed the files. The second asked for all six at once, and the budget ran out before the model could read them, so there was no answer. An assistant with a cap on its turns has to be told about the cap, or given a question it can finish within it, and a caller has to check for a missing answer as Chapter 5 checked `stop_reason`.

## Challenge 2. How each build started

From the transcripts, find the first two tool calls of each build and whether it ran the tests before writing anything.

In [7]:
# Challenge 2. Which tools did each build's first step use?
for n, (slug, t) in enumerate(transcripts.items(), 1):
    first = [s["tool"] for s in t["steps"] if s.get("tool")][:2]
    reads_tests_first = any(s.get("tool") == "Bash" and "pytest" in str(s.get("input")) for s in t["steps"][:3])
    print(f"Chapter {n}: first two tool calls {first}, ran the tests before writing: {reads_tests_first}")

Chapter 1: first two tool calls ['Read', 'Bash'], ran the tests before writing: True
Chapter 2: first two tool calls ['Read', 'Bash'], ran the tests before writing: True
Chapter 3: first two tool calls ['Read', 'Bash'], ran the tests before writing: True
Chapter 4: first two tool calls ['Read', 'Bash'], ran the tests before writing: True
Chapter 5: first two tool calls ['Read', 'Bash'], ran the tests before writing: False
Chapter 6: first two tool calls ['Read', 'Bash'], ran the tests before writing: False


Every build began with Read, for the spec, and then Bash. In the first four, that Bash call ran the failing tests. In Chapter 5's build it listed the folder, and in Chapter 6's it listed the folder and printed the tests, so neither ran the tests before writing. The same prompt pattern produced two orders of work, and the transcript is the only place to see which one a run followed.

## Challenge 3. Pin the gap with a test

Write a test that calls `triage()` with a stand-in client whose reply is `Billing.`, and expects `billing`.

In [8]:
# Challenge 3. A test that pins the gap the guide found, with a stand-in client and no API call.
from types import SimpleNamespace


class OneWordClient:
    # Answers every request with the same text, the way a model might punctuate a one-word label
    def __init__(self, text):
        self.messages = self
        self.text = text
    def create(self, **request):
        return SimpleNamespace(content=[SimpleNamespace(type="text", text=self.text)])


def test_triage_accepts_a_label_with_a_period():
    assert triage.triage(OneWordClient("Billing."), "I was charged twice.") == "billing"

try:
    test_triage_accepts_a_label_with_a_period()
    print("pass")
except AssertionError:
    print("FAIL: triage() returned", repr(triage.triage(OneWordClient("Billing."), "I was charged twice.")))

FAIL: triage() returned 'unrecognized'


The test fails, and `triage()` returns `unrecognized`. Added to Chapter 1's build as a given test, with a spec sentence that `triage()` normalizes the reply, it would make the next build call `normalize_label()` or fold its rule into `parse_category()`.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>